In [14]:
# ============================================================
# PDAC Clinical Trial Outcome Predictor
# Feature Engineering
# ============================================================

import pandas as pd
import numpy as np
import os

# --- Paths — adjust to your folder structure ---
DATA_DIR = '/Users/giuliaam/Desktop/Newlife/ML_Project/Clinical_Trial/Data/'
MASTER_FILE = '/Users/giuliaam/Desktop/Newlife/ML_Project/Clinical_Trial/pdac_master_labeled_v3.csv'

# --- Load master labeled dataset ---
master = pd.read_csv(MASTER_FILE)
print(f"Labeled trials loaded: {len(master)}")
print(f"SUCCESS: {(master['label']=='SUCCESS').sum()}")
print(f"FAILURE: {(master['label']=='FAILURE').sum()}")

# --- Load raw AACT tables we'll need for features ---
studies      = pd.read_csv(os.path.join(DATA_DIR, 'studies.txt'), sep='|', low_memory=False)
interventions = pd.read_csv(os.path.join(DATA_DIR, 'interventions.txt'), sep='|', low_memory=False)
eligibilities = pd.read_csv(os.path.join(DATA_DIR, 'eligibilities.txt'), sep='|', low_memory=False)
sponsors      = pd.read_csv(os.path.join(DATA_DIR, 'sponsors.txt'), sep='|', low_memory=False)
outcomes      = pd.read_csv(os.path.join(DATA_DIR, 'outcomes.txt'), sep='|', low_memory=False)

# Filter all tables to our 107 labeled trials only
our_ids = set(master['nct_id'])

interventions_pdac = interventions[interventions['nct_id'].isin(our_ids)]
eligibilities_pdac = eligibilities[eligibilities['nct_id'].isin(our_ids)]
sponsors_pdac      = sponsors[sponsors['nct_id'].isin(our_ids)]
outcomes_pdac      = outcomes[outcomes['nct_id'].isin(our_ids)]

print("\nTables loaded and filtered:")
print(f"Interventions rows: {len(interventions_pdac)}")
print(f"Eligibilities rows: {len(eligibilities_pdac)}")
print(f"Sponsors rows:      {len(sponsors_pdac)}")
print(f"Outcomes rows:      {len(outcomes_pdac)}")

# Quick look at interventions — this is our most important table
print("\nIntervention types:")
print(interventions_pdac['intervention_type'].value_counts())

Labeled trials loaded: 107
SUCCESS: 47
FAILURE: 60

Tables loaded and filtered:
Interventions rows: 273
Eligibilities rows: 107
Sponsors rows:      186
Outcomes rows:      358

Intervention types:
intervention_type
DRUG               208
BIOLOGICAL          21
RADIATION           18
OTHER               13
PROCEDURE           10
DIAGNOSTIC_TEST      2
DEVICE               1
Name: count, dtype: int64


In [15]:
# STRUCTURAL TRIAL DESIGN FEATURES
# Start with the master as our feature base:
# These come directly from the studies table — no NLP needed

features = master[['nct_id', 'label', 'phase', 'overall_status', 
                    'enrollment', 'start_date', 'completion_date']].copy()

# --- Phase encoding ---
# Ordinal encoding: Phase 2 < Phase 2/3 < Phase 3
phase_map = {
    'PHASE2': 2,
    'PHASE1/PHASE2': 1.5,
    'PHASE2/PHASE3': 2.5,
    'PHASE3': 3
}
features['phase_numeric'] = features['phase'].map(phase_map)

# --- Trial duration ---
features['start_date'] = pd.to_datetime(features['start_date'], errors='coerce')
features['completion_date'] = pd.to_datetime(features['completion_date'], errors='coerce')
features['duration_days'] = (features['completion_date'] - features['start_date']).dt.days

# --- Enrollment ---
# Log transform because enrollment is heavily skewed (some trials have 1000+ patients)
features['enrollment_log'] = np.log1p(features['enrollment'])

# --- Trial era ---
# Treatment landscape changed dramatically in PDAC around 2011 (FOLFIRINOX) 
# and 2013 (nab-paclitaxel). Encode this.
features['start_year'] = features['start_date'].dt.year
features['era'] = pd.cut(
    features['start_year'],
    bins=[0, 2011, 2015, 2020, 2030],
    labels=['pre_folfirinox', 'folfirinox_era', 'precision_era', 'modern']
)

print("Structured features built:")
print(features[['phase_numeric', 'duration_days', 'enrollment_log', 
                 'start_year', 'era']].describe())

print(f"\nMissing values:")
print(features.isnull().sum())

Structured features built:
       phase_numeric  duration_days  enrollment_log   start_year
count     107.000000      99.000000      102.000000   107.000000
mean        2.158879    1637.666667        4.188651  2008.532710
std         0.468831     849.915326        1.238803     6.566521
min         1.500000     184.000000        1.098612  1997.000000
25%         2.000000     941.000000        3.304929  2003.000000
50%         2.000000    1551.000000        3.960405  2008.000000
75%         2.000000    2141.000000        4.857867  2013.000000
max         3.000000    4731.000000        7.378384  2023.000000

Missing values:
nct_id             0
label              0
phase              0
overall_status     0
enrollment         5
start_date         0
completion_date    8
phase_numeric      0
duration_days      8
enrollment_log     5
start_year         0
era                0
dtype: int64


In [16]:
# INTERVENTION-BASED FEATURES
# Key question: what kind of drug, how many drugs, is there a combo?

# Number of interventions per trial (combination complexity)
n_interventions = interventions_pdac.groupby('nct_id').size().reset_index(name='n_interventions')

# Has drug component
has_drug = interventions_pdac[interventions_pdac['intervention_type']=='DRUG']\
    .groupby('nct_id').size().reset_index(name='n_drugs')

# Has biological (immunotherapy, antibody, etc.)
has_bio = interventions_pdac[interventions_pdac['intervention_type']=='BIOLOGICAL']\
    .groupby('nct_id')['nct_id'].count().reset_index(name='has_biological')
has_bio['has_biological'] = 1

# Has radiation
has_rad = interventions_pdac[interventions_pdac['intervention_type']=='RADIATION']\
    .groupby('nct_id')['nct_id'].count().reset_index(name='has_radiation')
has_rad['has_radiation'] = 1

# Merge into features
features = features.merge(n_interventions, on='nct_id', how='left')
features = features.merge(has_drug, on='nct_id', how='left')
features = features.merge(has_bio[['nct_id','has_biological']], on='nct_id', how='left')
features = features.merge(has_rad[['nct_id','has_radiation']], on='nct_id', how='left')

# Fill NaN with 0 for boolean features
features['has_biological'] = features['has_biological'].fillna(0)
features['has_radiation'] = features['has_radiation'].fillna(0)
features['n_drugs'] = features['n_drugs'].fillna(0)
features['n_interventions'] = features['n_interventions'].fillna(1)

# Combination therapy flag (more than one drug)
features['is_combination'] = (features['n_interventions'] > 1).astype(int)

print("Intervention features added:")
print(features[['n_interventions', 'n_drugs', 'has_biological', 
                 'has_radiation', 'is_combination']].describe())

Intervention features added:
       n_interventions     n_drugs  has_biological  has_radiation  \
count       107.000000  107.000000      107.000000     107.000000   
mean          2.551402    1.943925        0.140187       0.121495   
std           1.655372    1.219587        0.348815       0.328239   
min           1.000000    0.000000        0.000000       0.000000   
25%           1.500000    1.000000        0.000000       0.000000   
50%           2.000000    2.000000        0.000000       0.000000   
75%           3.000000    2.000000        0.000000       0.000000   
max          14.000000    7.000000        1.000000       1.000000   

       is_combination  
count      107.000000  
mean         0.747664  
std          0.436397  
min          0.000000  
25%          0.500000  
50%          1.000000  
75%          1.000000  
max          1.000000  


In [17]:
# SPONSOR TYPE
# Industry vs academic sponsor matters — industry has more resources 
# but also more selective about what they run (and when they stop)

lead_sponsors = sponsors_pdac[sponsors_pdac['lead_or_collaborator'] == 'lead'].copy()

lead_sponsors['is_industry'] = (
    lead_sponsors['agency_class'].str.upper() == 'INDUSTRY'
).astype(int)

features = features.merge(
    lead_sponsors[['nct_id', 'is_industry', 'agency_class']], 
    on='nct_id', how='left'
)

print("Sponsor type distribution:")
print(features['agency_class'].value_counts(dropna=False))
print(f"\nIndustry sponsored: {features['is_industry'].sum()}")
print(f"Academic/other:     {(features['is_industry']==0).sum()}")

Sponsor type distribution:
agency_class
OTHER        60
INDUSTRY     26
NETWORK      13
NIH           6
OTHER_GOV     2
Name: count, dtype: int64

Industry sponsored: 26
Academic/other:     81


In [18]:
# Get all unique drug names across our 107 trials
drug_list = interventions_pdac[
    interventions_pdac['intervention_type'].isin(['DRUG', 'BIOLOGICAL'])
]['name'].str.lower().str.strip().value_counts()

print(f"Unique drug/biological interventions: {len(drug_list)}")
print("\nMost common:")
print(drug_list.head(40))

Unique drug/biological interventions: 125

Most common:
name
gemcitabine hydrochloride                              24
gemcitabine                                            17
placebo                                                 8
fluorouracil                                            7
oxaliplatin                                             6
cisplatin                                               6
docetaxel                                               5
cetuximab                                               4
bevacizumab                                             4
capecitabine                                            4
nab-paclitaxel                                          4
leucovorin calcium                                      4
erlotinib hydrochloride                                 4
ipilimumab                                              3
irinotecan hydrochloride                                3
dexamethasone                                           3
pembrolizum

In [19]:
# MoA dictionary — maps drug name to mechanism of action
# Categories chosen specifically for PDAC biology

moa_dict = {
    # --- CHEMOTHERAPY BACKBONE ---
    # Nucleoside analogs — standard PDAC chemo since 1997
    'gemcitabine': 'chemo_nucleoside',
    'gemcitabine hydrochloride': 'chemo_nucleoside',
    'capecitabine': 'chemo_nucleoside',
    's1': 'chemo_nucleoside',
    
    # Fluoropyrimidines
    'fluorouracil': 'chemo_fluoropyrimidine',
    'leucovorin calcium': 'chemo_fluoropyrimidine',  # always used with 5-FU
    
    # Platinums — DNA crosslinkers
    'oxaliplatin': 'chemo_platinum',
    'cisplatin': 'chemo_platinum',
    'carboplatin': 'chemo_platinum',
    
    # Taxanes — microtubule stabilizers
    'paclitaxel': 'chemo_taxane',
    'nab-paclitaxel': 'chemo_taxane',
    'docetaxel': 'chemo_taxane',
    'ixabepilone': 'chemo_taxane',  # epothilone, same mechanism
    
    # Topoisomerase inhibitors
    'irinotecan': 'chemo_topoisomerase',
    'irinotecan hydrochloride': 'chemo_topoisomerase',
    'exatecan mesylate': 'chemo_topoisomerase',
    
    # Combinations — encoded separately as they imply multi-drug
    'folfirinox': 'chemo_combination',  # 5-FU/leucovorin/irinotecan/oxaliplatin
    'folfiri.3': 'chemo_combination',
    
    # Other chemo
    'glufosfamide': 'chemo_alkylating',
    
    # --- EGFR TARGETING ---
    # EGFR is mutated/overexpressed in PDAC but targeting it has largely failed
    'erlotinib': 'targeted_egfr',
    'erlotinib hydrochloride': 'targeted_egfr',
    'cetuximab': 'targeted_egfr',
    
    # --- VEGF/ANGIOGENESIS ---
    # Anti-angiogenic — largely failed in PDAC due to poor vascularization
    'bevacizumab': 'targeted_vegf',
    
    # --- IMMUNOTHERAPY ---
    # Checkpoint inhibitors — largely failed in PDAC due to cold tumor microenvironment
    'pembrolizumab': 'immuno_checkpoint',
    'nivolumab': 'immuno_checkpoint',
    'ipilimumab': 'immuno_checkpoint',
    
    # Other immunotherapy
    'wild-type reovirus': 'immuno_oncolytic',
    'dc-cik treatment': 'immuno_cell_therapy',
    
    # --- mTOR PATHWAY ---
    'rad001': 'targeted_mtor',  # everolimus
    
    # --- STROMA TARGETING ---
    # PDAC stroma is dense and immunosuppressive — major research focus
    'necuparanib': 'stroma_targeting',
    
    # --- ANTI-INFLAMMATORY ---
    'indomethacin': 'anti_inflammatory',
    'dexamethasone': 'anti_inflammatory',
    
    # --- IL-6/JAK-STAT ---
    # Relevant to PDAC immunosuppression
    'tocilizumab': 'targeted_il6',
    
    # --- ANTICOAGULANT ---
    # Tested in PDAC due to hypercoagulability
    'enoxaparin': 'anticoagulant',
    
    # --- OTHER/UNCLEAR ---
    'opioid growth factor': 'other',
    'metenkephalin, ogf-opioid growth factor': 'other',
    'drug gc4711': 'other',
    'influenza vaccine': 'other',
    
    # --- CONTROL ---
    'placebo': 'control',
}

# Now apply to each trial
# For each trial, get the set of MoA categories present

def get_trial_moa(nct_id):
    drugs = interventions_pdac[
        (interventions_pdac['nct_id'] == nct_id) &
        (interventions_pdac['intervention_type'].isin(['DRUG', 'BIOLOGICAL']))
    ]['name'].str.lower().str.strip().tolist()
    
    moas = set()
    for drug in drugs:
        moa = moa_dict.get(drug, 'unknown')
        moas.add(moa)
    return list(moas)

# Apply
features['moa_list'] = features['nct_id'].apply(get_trial_moa)

# Create binary columns for each MoA category
moa_categories = [
    'chemo_nucleoside', 'chemo_fluoropyrimidine', 'chemo_platinum',
    'chemo_taxane', 'chemo_topoisomerase', 'chemo_combination',
    'chemo_alkylating', 'targeted_egfr', 'targeted_vegf',
    'immuno_checkpoint', 'immuno_oncolytic', 'immuno_cell_therapy',
    'targeted_mtor', 'stroma_targeting', 'targeted_il6',
    'anti_inflammatory', 'anticoagulant', 'other', 'unknown'
]

for cat in moa_categories:
    features[f'moa_{cat}'] = features['moa_list'].apply(
        lambda x: 1 if cat in x else 0
    )

# Check coverage
print("MoA feature coverage:")
for cat in moa_categories:
    n = features[f'moa_{cat}'].sum()
    if n > 0:
        print(f"  {cat}: {n} trials")

# How many trials have at least one unknown MoA?
print(f"\nTrials with unknown drug: {features['moa_unknown'].sum()}")
print(f"Trials with no MoA mapped: {(features['moa_list'].apply(len)==0).sum()}")

MoA feature coverage:
  chemo_nucleoside: 42 trials
  chemo_fluoropyrimidine: 7 trials
  chemo_platinum: 13 trials
  chemo_taxane: 13 trials
  chemo_topoisomerase: 6 trials
  chemo_combination: 3 trials
  chemo_alkylating: 1 trials
  targeted_egfr: 9 trials
  targeted_vegf: 4 trials
  immuno_checkpoint: 6 trials
  immuno_oncolytic: 2 trials
  immuno_cell_therapy: 1 trials
  targeted_mtor: 2 trials
  stroma_targeting: 1 trials
  targeted_il6: 2 trials
  anti_inflammatory: 4 trials
  anticoagulant: 2 trials
  other: 3 trials
  unknown: 59 trials

Trials with unknown drug: 59
Trials with no MoA mapped: 2


In [20]:
moa_dict_extended = {
    # --- IMAGING/DIAGNOSTIC — exclude from MoA (not therapeutic) ---
    '18f-fludeoxyglucose (18f-fdg)': 'diagnostic',
    '18f-fpprgd2': 'diagnostic',
    'cetuximab-irdye800': 'diagnostic',

    # --- CHEMO VARIANTS (same MoA as existing entries) ---
    '5fluorouracil': 'chemo_fluoropyrimidine',
    'calcium leucovorin': 'chemo_fluoropyrimidine',
    'lv5fu2': 'chemo_fluoropyrimidine',       # leucovorin/5-FU regimen
    'folfox6': 'chemo_combination',            # 5-FU/leucovorin/oxaliplatin
    'gemcitabine (chemotherapy)': 'chemo_nucleoside',
    'gemcitabine alone': 'chemo_nucleoside',
    'gemcitabine + capecitabine': 'chemo_nucleoside',
    'albumin-bound paclitaxel (abi-007)': 'chemo_taxane',   # nab-paclitaxel brand
    'paclitaxel-loaded polymeric micelle': 'chemo_taxane',
    'nab-paclitaxel+ gemcitabine': 'chemo_taxane',
    'liposomal cisplatin': 'chemo_platinum',
    'pemetrexed': 'chemo_antimetabolite',      # antifolate
    'mitomycin c': 'chemo_alkylating',
    'cyclophosphamide': 'chemo_alkylating',
    'fludarabine': 'chemo_nucleoside',
    'th-302': 'chemo_alkylating',              # hypoxia-activated prodrug alkylator
    'mkc-1': 'chemo_antimetabolite',           # dual tubulin/importin inhibitor

    # --- EGFR/HER2 TARGETING ---
    'erlotinib 100mg tab': 'targeted_egfr',
    'gemcitabine - trastuzumab - erlotinib': 'targeted_egfr',  # combination
    'trastuzumab': 'targeted_her2',            # HER2 — rarely amplified in PDAC
    'immu-107': 'targeted_egfr',               # anti-EGFR antibody

    # --- VEGF/ANGIOGENESIS ---
    'aflibercept (ziv-aflibercept, ave0005, vegf trap, zaltrap®)': 'targeted_vegf',
    'rhumab-vegf': 'targeted_vegf',            # bevacizumab precursor
    'lenvatinib': 'targeted_vegf',             # multi-RTK inhibitor inc VEGFR
    'nintedanib': 'targeted_vegf',             # VEGFR/FGFR/PDGFR inhibitor
    'cabozantinib 40 mg': 'targeted_vegf',     # MET/VEGFR2 inhibitor

    # --- RAS/MAPK PATHWAY ---
    # KRAS drives >95% of PDAC — these target downstream effectors
    'binimetinib': 'targeted_ras_mapk',        # MEK inhibitor
    'tipifarnib': 'targeted_ras_mapk',         # farnesyl transferase inhibitor (old KRAS strategy)
    'r115777': 'targeted_ras_mapk',            # farnesyltransferase inhibitor
    'isis 2503': 'targeted_ras_mapk',          # antisense oligo targeting H-RAS
    'on 01910.na': 'targeted_ras_mapk',        # Plk1/PI3K inhibitor

    # --- PI3K/AKT/mTOR PATHWAY ---
    'everolimus': 'targeted_mtor',             # mTOR inhibitor (= RAD001)
    'rad001': 'targeted_mtor',
    'sirolimus': 'targeted_mtor',              # rapamycin — mTOR inhibitor
    'perifosine': 'targeted_pi3k_akt',         # AKT inhibitor
    'bez235': 'targeted_pi3k_akt',             # dual PI3K/mTOR inhibitor

    # --- HEDGEHOG PATHWAY ---
    # Dense stroma in PDAC driven partly by Hh signaling — largely failed
    'vismodegib': 'targeted_hedgehog',
    'ipi-926': 'targeted_hedgehog',

    # --- STROMA TARGETING ---
    'pegph20': 'stroma_targeting',             # hyaluronidase — degrades HA in stroma
    'necuparanib': 'stroma_targeting',
    'masitinib': 'stroma_targeting',           # c-Kit/PDGFR — targets stroma signaling
    'verteporfin': 'stroma_targeting',         # YAP inhibitor — CAF signaling
    'glyceryl nitrate': 'stroma_targeting',    # NO donor — vascular/stroma normalization

    # --- PARP INHIBITORS ---
    # Relevant in BRCA-mutated PDAC (~5-7%)
    'olaparib': 'targeted_parp',
    'talazoparib': 'targeted_parp',
    'niraparib': 'targeted_parp',

    # --- IMMUNOTHERAPY ---
    'avelumab': 'immuno_checkpoint',           # anti-PD-L1
    'durvalumab': 'immuno_checkpoint',         # anti-PD-L1
    'tremelimumab': 'immuno_checkpoint',       # anti-CTLA4
    'gv1001': 'immuno_vaccine',                # telomerase peptide vaccine
    'cancer stem cell vaccine': 'immuno_vaccine',
    'npc-1c': 'immuno_vaccine',                # anti-MUC5AC antibody
    'interleukin-2': 'immuno_cytokine',
    'autologous tumor infiltrating lymphocytes mda-til': 'immuno_cell_therapy',
    'amg 479': 'targeted_igf1r',               # IGF-1R antibody — immune/growth

    # --- AUTOPHAGY ---
    # PDAC is highly autophagy-dependent — active research area
    'hydroxychloroquine': 'targeted_autophagy', # chloroquine derivative, autophagy inhibitor
    'digoxin': 'targeted_autophagy',            # HIF-1a/autophagy inhibitor

    # --- SRC/ABL KINASE ---
    'azd0530': 'targeted_src',                 # Src/Abl inhibitor
    'imatinib mesylate': 'targeted_src',       # BCR-Abl/c-Kit/PDGFR

    # --- ANTI-INFLAMMATORY/IMMUNOSUPPRESSIVE ---
    'infliximab; gemcitabine': 'anti_inflammatory',  # anti-TNF
    'etanercept': 'anti_inflammatory',               # anti-TNF
    'indomethacin': 'anti_inflammatory',
    'dipyridamole': 'anti_inflammatory',             # phosphodiesterase inhibitor
    'methylprednisolone': 'anti_inflammatory',
    'mycophenolate mofetil': 'anti_inflammatory',    # immunosuppressive
    'anti-thymocyte globulin (rabbit)': 'anti_inflammatory',
    'belatacept': 'anti_inflammatory',
    'tacrolimus': 'anti_inflammatory',
    'bryostatin 1': 'anti_inflammatory',             # PKC modulator

    # --- SOMATOSTATIN ANALOGS ---
    # Neuroendocrine signaling — more relevant to PNET but tested in PDAC
    'octreotide': 'somatostatin_analog',
    'somatostatin': 'somatostatin_analog',
    'lanreotide (autogel formulation)': 'somatostatin_analog',
    'vapreotide': 'somatostatin_analog',

    # --- HSP90 INHIBITOR ---
    'sta-9090': 'targeted_hsp90',              # ganetespib — chaperone inhibitor

    # --- ADC / NOVEL TARGETED ---
    'sacituzumab govitecan-hziy (sg)': 'targeted_adc',   # TROP2-directed ADC
    'zilovertamab vedotin': 'targeted_adc',               # ROR1-directed ADC

    # --- SUPPORTIVE / NON-ONCOLOGY ---
    'pancrelipase delayed release': 'supportive',  # enzyme replacement, not anti-tumor
    'ferinject (ferric carboxymaltose)': 'supportive',  # iron infusion

    # --- UNCLEAR / MILESTONE LABELS (not real drugs) ---
    'milestone 1: targeted chemotherapy prior to surgery': 'chemo_combination',
    'milestone 3: targeted chemotherapy prior to surgery': 'chemo_combination',
    'milestone 4: standard folfirinox chemotherapy prior to surgery': 'chemo_combination',
    'milestone 5: targeted chemotherapy after surgery': 'chemo_combination',
    'milestone 6: gemcitabine after surgery': 'chemo_nucleoside',
    'milestone 8: targeted chemotherapy after surgery': 'chemo_combination',
    'milestone 9: gemcitabine after surgery': 'chemo_nucleoside',

    # --- OTHER ---
    'photodynamic therapy': 'other',
    'placebo comparator': 'control',
    'tak-500': 'other',                        # STING agonist — immunotherapy adjacent
    'gemcitabine + capecitabine': 'chemo_nucleoside',
}

# Merge with original dictionary
moa_dict.update(moa_dict_extended)

# Updated MoA categories — add new ones
moa_categories_updated = [
    'chemo_nucleoside', 'chemo_fluoropyrimidine', 'chemo_platinum',
    'chemo_taxane', 'chemo_topoisomerase', 'chemo_combination',
    'chemo_alkylating', 'chemo_antimetabolite',
    'targeted_egfr', 'targeted_her2', 'targeted_vegf',
    'targeted_ras_mapk', 'targeted_pi3k_akt', 'targeted_mtor',
    'targeted_hedgehog', 'targeted_parp', 'targeted_igf1r',
    'targeted_src', 'targeted_hsp90', 'targeted_adc', 'targeted_il6',
    'targeted_autophagy',
    'immuno_checkpoint', 'immuno_oncolytic', 'immuno_cell_therapy',
    'immuno_vaccine', 'immuno_cytokine',
    'stroma_targeting', 'anti_inflammatory', 'anticoagulant',
    'somatostatin_analog', 'supportive', 'diagnostic',
    'other', 'control', 'unknown'
]

# Recompute MoA features with full dictionary
features['moa_list'] = features['nct_id'].apply(get_trial_moa)

for cat in moa_categories_updated:
    features[f'moa_{cat}'] = features['moa_list'].apply(
        lambda x: 1 if cat in x else 0
    )

print("Updated MoA coverage:")
for cat in moa_categories_updated:
    n = features[f'moa_{cat}'].sum()
    if n > 0:
        print(f"  {cat}: {n} trials")

print(f"\nTrials still with unknown drug: {features['moa_unknown'].sum()}")

Updated MoA coverage:
  chemo_nucleoside: 46 trials
  chemo_fluoropyrimidine: 9 trials
  chemo_platinum: 14 trials
  chemo_taxane: 16 trials
  chemo_topoisomerase: 6 trials
  chemo_combination: 5 trials
  chemo_alkylating: 4 trials
  chemo_antimetabolite: 2 trials
  targeted_egfr: 12 trials
  targeted_her2: 1 trials
  targeted_vegf: 9 trials
  targeted_ras_mapk: 5 trials
  targeted_pi3k_akt: 2 trials
  targeted_mtor: 4 trials
  targeted_hedgehog: 1 trials
  targeted_parp: 2 trials
  targeted_igf1r: 1 trials
  targeted_src: 2 trials
  targeted_hsp90: 1 trials
  targeted_adc: 2 trials
  targeted_il6: 2 trials
  targeted_autophagy: 2 trials
  immuno_checkpoint: 8 trials
  immuno_oncolytic: 2 trials
  immuno_cell_therapy: 2 trials
  immuno_vaccine: 3 trials
  immuno_cytokine: 1 trials
  stroma_targeting: 5 trials
  anti_inflammatory: 9 trials
  anticoagulant: 2 trials
  somatostatin_analog: 3 trials
  supportive: 2 trials
  diagnostic: 2 trials
  other: 5 trials
  control: 9 trials

Trials

In [21]:
# Look at the eligibility table structure
print(eligibilities_pdac.columns.tolist())
print(eligibilities_pdac.head(3))
print(f"\nRows: {len(eligibilities_pdac)}")

['id', 'nct_id', 'sampling_method', 'gender', 'minimum_age', 'maximum_age', 'healthy_volunteers', 'population', 'criteria', 'gender_description', 'gender_based', 'adult', 'child', 'older_adult']
             id       nct_id sampling_method gender minimum_age maximum_age  \
1727  189780288  NCT00058201             NaN    ALL    18 Years         NaN   
5576  189896105  NCT00014651             NaN    ALL    18 Years    90 Years   
7167  190010470  NCT00265876             NaN    ALL    18 Years   120 Years   

     healthy_volunteers population  \
1727                  f        NaN   
5576                  f        NaN   
7167                  f        NaN   

                                               criteria gender_description  \
1727  DISEASE CHARACTERISTICS:~* Histologically conf...                NaN   
5576  DISEASE CHARACTERISTICS: Planned elective panc...                NaN   
7167  DISEASE CHARACTERISTICS:~* Histologically or c...                NaN   

     gender_based adul

In [22]:
# The eligibility text is usually one big block per trial
# We want to extract specific biological signals from it
import re
def extract_eligibility_features(text):
    """
    Extract binary features from eligibility criteria text.
    Each feature = a biological/clinical assumption encoded in the protocol.
    """
    if pd.isna(text):
        return {}
    
    t = text.lower()
    
    features = {
        # --- Biomarker selection ---
        # Does the trial require a specific molecular marker to enroll?
        # Biomarker-selected trials are more likely to succeed (precision medicine)
        'elig_kras_required': bool(re.search(r'kras', t)),
        'elig_brca_required': bool(re.search(r'brca', t)),
        'elig_pdl1_required': bool(re.search(r'pd.?l1', t)),
        'elig_her2_required': bool(re.search(r'her2|erbb2', t)),
        'elig_msi_required':  bool(re.search(r'msi|microsatellite', t)),
        'elig_biomarker_any': bool(re.search(
            r'kras|brca|pd.?l1|her2|msi|microsatellite|biomarker|mutation|expression', t)),

        # --- Disease stage ---
        'elig_metastatic': bool(re.search(r'metastatic|stage iv|stage 4', t)),
        'elig_locally_advanced': bool(re.search(r'locally advanced|unresectable', t)),
        'elig_resectable': bool(re.search(r'resectable|resection|surgery', t)),
        'elig_adjuvant': bool(re.search(r'adjuvant|post.?surgical|post.?resection', t)),

        # --- Performance status ---
        # ECOG 0-1 = stricter = healthier patients = better outcomes
        'elig_ecog_strict': bool(re.search(r'ecog.{0,10}[01]|performance status.{0,10}[01]', t)),

        # --- Prior therapy requirements ---
        'elig_first_line': bool(re.search(r'first.?line|no prior.{0,30}(chemo|treatment|therapy)', t)),
        'elig_second_line': bool(re.search(r'second.?line|previously treated|prior.{0,30}(chemo|treatment)', t)),

        # --- CA19-9 ---
        # Common PDAC biomarker used for enrollment or stratification
        'elig_ca199': bool(re.search(r'ca.?19.?9|ca19', t)),
    }
    
    return features

# Apply to all trials
elig_features_list = eligibilities_pdac['criteria'].apply(extract_eligibility_features)
elig_df = pd.DataFrame(elig_features_list.tolist())
elig_df['nct_id'] = eligibilities_pdac['nct_id'].values

# Convert booleans to int
bool_cols = [c for c in elig_df.columns if c != 'nct_id']
elig_df[bool_cols] = elig_df[bool_cols].astype(int)

# Merge into features
features = features.merge(elig_df, on='nct_id', how='left')

print("Eligibility features distribution:")
for col in bool_cols:
    n = features[col].sum()
    print(f"  {col}: {n} trials ({n/len(features)*100:.0f}%)")

Eligibility features distribution:
  elig_kras_required: 2 trials (2%)
  elig_brca_required: 0 trials (0%)
  elig_pdl1_required: 5 trials (5%)
  elig_her2_required: 1 trials (1%)
  elig_msi_required: 2 trials (2%)
  elig_biomarker_any: 8 trials (7%)
  elig_metastatic: 79 trials (74%)
  elig_locally_advanced: 56 trials (52%)
  elig_resectable: 82 trials (77%)
  elig_adjuvant: 45 trials (42%)
  elig_ecog_strict: 71 trials (66%)
  elig_first_line: 47 trials (44%)
  elig_second_line: 76 trials (71%)
  elig_ca199: 4 trials (4%)


In [23]:
# Fix the resectable and second_line regex — 
# exclude when preceded by "un" or "non"
def extract_eligibility_features(text):
    if pd.isna(text):
        return {}
    
    t = text.lower()
    
    features = {
        'elig_kras_required':    bool(re.search(r'kras', t)),
        'elig_brca_required':    bool(re.search(r'brca[12]?|brca (1|2|mutation)', t)),
        'elig_pdl1_required':    bool(re.search(r'pd.?l1', t)),
        'elig_her2_required':    bool(re.search(r'her2|erbb2', t)),
        'elig_msi_required':     bool(re.search(r'msi|microsatellite', t)),
        'elig_biomarker_any':    bool(re.search(
            r'kras|brca|pd.?l1|her2|msi|microsatellite|biomarker|mutation required|expression required', t)),

        # Resectable — only if NOT preceded by un/non
        'elig_resectable':       bool(re.search(r'(?<!un)(?<!non.)resectable', t)),
        'elig_unresectable':     bool(re.search(r'unresectable|non.?resectable', t)),
        'elig_metastatic':       bool(re.search(r'metastatic|stage iv|stage 4', t)),
        'elig_locally_advanced': bool(re.search(r'locally advanced', t)),
        'elig_adjuvant':         bool(re.search(r'adjuvant|post.?surgical|post.?resection', t)),

        'elig_ecog_strict':   bool(re.search(r'ecog.{0,10}[01]|performance status.{0,10}[01]', t)),

        # First line — no prior chemo
        'elig_first_line':    bool(re.search(
            r'first.?line|no prior.{0,30}(chemo|treatment|therapy)|treatment.?naive', t)),
        # Second line — previously treated
        'elig_second_line':   bool(re.search(
            r'second.?line|previously treated|prior.{0,30}(gemcitabine|chemo|treatment)', t)),

        'elig_ca199':         bool(re.search(r'ca.?19.?9|ca19', t)),
    }
    
    return features

# Recompute
elig_features_list = eligibilities_pdac['criteria'].apply(extract_eligibility_features)
elig_df = pd.DataFrame(elig_features_list.tolist())
elig_df['nct_id'] = eligibilities_pdac['nct_id'].values
bool_cols = [c for c in elig_df.columns if c != 'nct_id']
elig_df[bool_cols] = elig_df[bool_cols].astype(int)

print("Refined eligibility features:")
for col in bool_cols:
    n = elig_df[col].sum()
    print(f"  {col}: {n} trials ({n/len(elig_df)*100:.0f}%)")

Refined eligibility features:
  elig_kras_required: 2 trials (2%)
  elig_brca_required: 0 trials (0%)
  elig_pdl1_required: 5 trials (5%)
  elig_her2_required: 1 trials (1%)
  elig_msi_required: 2 trials (2%)
  elig_biomarker_any: 7 trials (7%)
  elig_resectable: 9 trials (8%)
  elig_unresectable: 34 trials (32%)
  elig_metastatic: 79 trials (74%)
  elig_locally_advanced: 40 trials (37%)
  elig_adjuvant: 45 trials (42%)
  elig_ecog_strict: 71 trials (66%)
  elig_first_line: 47 trials (44%)
  elig_second_line: 77 trials (72%)
  elig_ca199: 4 trials (4%)


In [24]:
# Drop old elig columns from features and re-merge
old_elig_cols = [c for c in features.columns if c.startswith('elig_')]
features = features.drop(columns=old_elig_cols)
features = features.merge(elig_df, on='nct_id', how='left')

print(f"\nFinal feature matrix shape: {features.shape}")
print(f"Columns: {features.columns.tolist()}")


Final feature matrix shape: (107, 71)
Columns: ['nct_id', 'label', 'phase', 'overall_status', 'enrollment', 'start_date', 'completion_date', 'phase_numeric', 'duration_days', 'enrollment_log', 'start_year', 'era', 'n_interventions', 'n_drugs', 'has_biological', 'has_radiation', 'is_combination', 'is_industry', 'agency_class', 'moa_list', 'moa_chemo_nucleoside', 'moa_chemo_fluoropyrimidine', 'moa_chemo_platinum', 'moa_chemo_taxane', 'moa_chemo_topoisomerase', 'moa_chemo_combination', 'moa_chemo_alkylating', 'moa_targeted_egfr', 'moa_targeted_vegf', 'moa_immuno_checkpoint', 'moa_immuno_oncolytic', 'moa_immuno_cell_therapy', 'moa_targeted_mtor', 'moa_stroma_targeting', 'moa_targeted_il6', 'moa_anti_inflammatory', 'moa_anticoagulant', 'moa_other', 'moa_unknown', 'moa_chemo_antimetabolite', 'moa_targeted_her2', 'moa_targeted_ras_mapk', 'moa_targeted_pi3k_akt', 'moa_targeted_hedgehog', 'moa_targeted_parp', 'moa_targeted_igf1r', 'moa_targeted_src', 'moa_targeted_hsp90', 'moa_targeted_adc', '

In [31]:
import re

def extract_biophysical_features(text):
    if pd.isna(text) or text.strip() == '':
        return {}
    
    t = text.lower()
    
    return {
        # --- Stiffness / mechanics ---
        # Does the trial mention tumor mechanical properties?
        'bio_stiffness': bool(re.search(
            r'stiffness|rigidity|elastic modulus|mechanical propert|viscoelast', t)),
        
        # --- Extracellular matrix ---
        'bio_ecm': bool(re.search(
            r'extracellular matrix|ecm\b|matrix remodel|fibronectin|laminin', t)),
        
        # --- Collagen ---
        'bio_collagen': bool(re.search(
            r'collagen|fibrosis|desmoplast', t)),
        
        # --- Hyaluronic acid / HA ---
        # PEGPH20 targets HA — already in stroma MoA but worth capturing in text too
        'bio_hyaluronan': bool(re.search(
            r'hyaluronan|hyaluronic acid|\bha\b.{0,20}stroma|hyal', t)),
        
        # --- Cancer associated fibroblasts ---
        'bio_caf': bool(re.search(
            r'cancer.associated fibroblast|\bcaf\b|stellate cell|activated fibroblast', t)),
        
        # --- Stroma broadly ---
        'bio_stroma': bool(re.search(
            r'stroma|stromal|desmoplast|tumor microenvironment|tme\b', t)),
        
        # --- Interstitial pressure ---
        # High IFP in PDAC prevents drug delivery
        'bio_pressure': bool(re.search(
            r'interstitial (fluid )?pressure|ifp\b|tumor pressure', t)),
        
        # --- Vascularization / perfusion ---
        'bio_vascular': bool(re.search(
            r'vascular(ization)?|perfusion|blood flow|hypovascular|poorly vascular', t)),
        
        # --- Hypoxia ---
        # PDAC is severely hypoxic due to poor vascularization
        'bio_hypoxia': bool(re.search(
            r'hypox|oxygen.{0,20}(level|tension|deplet)|hif.1', t)),
    }

In [32]:
# ============================================================
# BIOPHYSICAL FEATURES — TME & Stroma
# ============================================================

brief_summaries = pd.read_csv(os.path.join(DATA_DIR, 'brief_summaries.txt'), 
                               sep='|', low_memory=False)
detailed_descriptions = pd.read_csv(os.path.join(DATA_DIR, 'detailed_descriptions.txt'), 
                                     sep='|', low_memory=False)

summaries_pdac = brief_summaries[brief_summaries['nct_id'].isin(our_ids)]
details_pdac = detailed_descriptions[detailed_descriptions['nct_id'].isin(our_ids)]

print(f"Trials with brief summary: {summaries_pdac['nct_id'].nunique()}")
print(f"Trials with detailed description: {details_pdac['nct_id'].nunique()}")

# Combine both text fields per trial
text_combined = summaries_pdac[['nct_id', 'description']].merge(
    details_pdac[['nct_id', 'description']].rename(
        columns={'description': 'detailed'}),
    on='nct_id', how='outer'
)

text_combined['full_text'] = (
    text_combined['description'].fillna('') + ' ' +
    text_combined['detailed'].fillna('')
).str.lower().str.strip()

print(f"Trials with any text: {(text_combined['full_text'].str.len() > 10).sum()}")
print(f"\nSample (first 300 chars):")
print(text_combined['full_text'].iloc[0][:300])

Trials with brief summary: 107
Trials with detailed description: 107
Trials with any text: 107

Sample (first 300 chars):
rationale: drugs used in chemotherapy use different ways to stop tumor cells from dividing so they stop growing or die. combining more than one drug may kill more tumor cells. chemotherapy following surgery may be an effective treatment for pancreatic cancer.~purpose: phase ii trial to study the eff


In [25]:
# --- Final feature matrix assembly ---

# Columns to use for modeling — drop metadata we don't want the model to see
drop_cols = [
    'label',           # target — keep separate
    'nct_id',          # identifier
    'phase',           # replaced by phase_numeric
    'overall_status',  # all completed/terminated — no signal
    'start_date',      # replaced by start_year and era
    'completion_date', # replaced by duration_days
    'moa_list',        # list format — replaced by binary columns
    'agency_class',    # replaced by is_industry
    'why_stopped',     # text — not a feature
    'reasoning',       # text from mistral
    'primary_endpoint',# text from mistral
    'title',           # text
    'abstract',        # text
    'pmid',            # identifier
    'source',          # labeling source — not a real feature
    'confidence',      # labeling metadata
]

# Keep only columns that exist
drop_cols = [c for c in drop_cols if c in features.columns]

X = features.drop(columns=drop_cols)
y = (features['label'] == 'SUCCESS').astype(int)  # 1=success, 0=failure

print(f"Feature matrix X: {X.shape}")
print(f"Target y: {y.shape}")
print(f"Class balance: {y.sum()} success ({y.mean()*100:.0f}%), {(1-y).sum()} failure ({(1-y.mean())*100:.0f}%)")

# Check for remaining non-numeric columns
non_numeric = X.select_dtypes(exclude=[np.number]).columns.tolist()
print(f"\nNon-numeric columns to handle: {non_numeric}")

# Check missing values
missing = X.isnull().sum()
missing = missing[missing > 0]
print(f"\nMissing values:")
print(missing)

Feature matrix X: (107, 63)
Target y: (107,)
Class balance: 47 success (44%), 60 failure (56%)

Non-numeric columns to handle: ['era']

Missing values:
enrollment        5
duration_days     8
enrollment_log    5
dtype: int64


In [26]:
# Handle non-numeric and missing values

# Era — encode as ordinal
era_map = {'pre_folfirinox': 0, 'folfirinox_era': 1, 'precision_era': 2, 'modern': 3}
if 'era' in X.columns:
    X['era'] = X['era'].map(era_map)

# Fill remaining missing values
# Enrollment and duration — fill with median (not mean, skewed distributions)
for col in ['enrollment', 'enrollment_log', 'duration_days']:
    if col in X.columns:
        X[col] = X[col].fillna(X[col].median())

# Boolean features — fill with 0
X = X.fillna(0)

# Verify
print(f"Final X shape: {X.shape}")
print(f"Any remaining NaN: {X.isnull().any().any()}")
print(f"All numeric: {X.select_dtypes(exclude=[np.number]).shape[1] == 0}")

# Save
X.to_csv('pdac_features_X.csv', index=False)
y.to_csv('pdac_labels_y.csv', index=False)
print("\nSaved X and y.")

Final X shape: (107, 63)
Any remaining NaN: False
All numeric: False

Saved X and y.


In [29]:
y.head()

0    1
1    1
2    0
3    1
4    1
Name: label, dtype: int64